In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import pandas as pd
from tqdm import tqdm  
from torch.cuda.amp import GradScaler, autocast


def prepare_dataset(base_dir):
    data = []
    labels = []

   
    for subject in os.listdir(base_dir):
        subject_path = os.path.join(base_dir, subject)

        
        if os.path.isdir(subject_path):
            for category in ['fall', 'non_fall']:
                category_path = os.path.join(subject_path, category)

                
                if os.path.isdir(category_path):
                    for sub_category in os.listdir(category_path):
                        sub_category_path = os.path.join(category_path, sub_category)

                        
                        if os.path.isdir(sub_category_path):
                            for img_file in os.listdir(sub_category_path):
                                img_path = os.path.join(sub_category_path, img_file)

                                
                                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                                    
                                    try:
                                        with Image.open(img_path) as img:
                                            img.verify()  
                                        data.append(img_path)
                                        if category == 'fall':
                                            labels.append(1)  
                                        elif category == 'non_fall':
                                            labels.append(0)  
                                    except (IOError, SyntaxError) as e:
                                        print(f"File {img_path} tidak dapat dibuka dan akan diabaikan. Error: {e}")
    return data, labels


train_transform = transforms.Compose([
    transforms.Resize(224),  
    transforms.RandomHorizontalFlip(),  
    transforms.RandomRotation(10),      
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),  
])


test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])


class CustomDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, label


class TestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.image_names = sorted(os.listdir(image_dir))  
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_name


def get_data_loaders(train_dir, test_dir, batch_size=32, val_split=0.2, num_workers=0, pin_memory=False):
    
    train_data, train_labels = prepare_dataset(train_dir)

    print(f"Jumlah data pelatihan: {len(train_data)}")
    print(f"Jumlah label pelatihan: {len(train_labels)}")

   
    full_dataset = CustomDataset(train_data, train_labels, transform=train_transform)

  
    train_size = int((1 - val_split) * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    print(f"Ukuran dataset train: {train_size}, Ukuran dataset val: {val_size}")

   
    val_dataset.dataset.transform = test_transform

    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

   
    test_dataset = TestDataset(test_dir, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

    print(f"Jumlah data test: {len(test_dataset)}")

    return train_loader, val_loader, test_loader


def initialize_model(num_classes=2, pretrained=True):
    model = models.vit_b_16(pretrained=pretrained)


    if isinstance(model.heads, nn.Sequential):
        
        num_features = model.heads[-1].in_features
        model.heads = nn.Sequential(
            nn.Dropout(0.5),  
            nn.Linear(num_features, num_classes)
        )
    elif isinstance(model.heads, nn.Linear):
        num_features = model.heads.in_features
        model.heads = nn.Linear(num_features, num_classes)
    else:
        raise NotImplementedError("Struktur model.heads tidak dikenali. Silakan periksa kembali struktur model.")

    return model


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler=None, num_epochs=10, device='cuda'):
    best_val_accuracy = 0.0
    scaler = GradScaler()

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        
        model.train()
        running_loss = 0.0

        
        for inputs, labels in tqdm(train_loader, desc="Training", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

           
            optimizer.zero_grad()

            
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

           
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        
        model.eval()
        running_val_loss = 0.0
        running_corrects = 0

        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc="Validation", leave=False):
                inputs = inputs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)
                running_val_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = running_corrects.double() / len(val_loader.dataset)

        print(f"Train Loss: {epoch_loss:.4f} Val Loss: {epoch_val_loss:.4f} Val Acc: {epoch_val_acc:.4f}")

        
        if epoch_val_acc > best_val_accuracy:
            best_val_accuracy = epoch_val_acc
            torch.save(model.state_dict(), "best_vit_model.pth")
            print("Model terbaik disimpan!")

        
        if scheduler:
            scheduler.step()

    print("Pelatihan selesai!")


def predict_and_create_submission(model, test_loader, device='cuda', submission_file='sample_submission.csv'):
    model.eval()
    predictions = []
    ids = []

    with torch.no_grad():
        for inputs, img_names in tqdm(test_loader, desc="Predicting", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()  
            predictions.extend(preds)
            ids.extend(img_names)

   
    submission = pd.DataFrame({
        "id": ids,
        "label": predictions  
    })

    
    submission.sort_values(by="id", inplace=True)

   
    submission.to_csv(submission_file, index=False)
    print(f"File {submission_file} berhasil dibuat!")


def main():
    
    train_dir = "C:/Users/ijalg/Downloads/Data Slayer/train"
    test_dir = "C:/Users/ijalg/Downloads/Data Slayer/test"

    
    batch_size = 32
    num_epochs = 5
    learning_rate = 1e-4
    val_split = 0.2
    num_workers = 0  
    pin_memory = False  

   
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Penggunaan device: {device}")

    
    train_loader, val_loader, test_loader = get_data_loaders(
        train_dir=train_dir,
        test_dir=test_dir,
        batch_size=batch_size,
        val_split=val_split,
        num_workers=num_workers,
        pin_memory=pin_memory
    )

    
    model = initialize_model(num_classes=2, pretrained=True)
    model = model.to(device)

    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

   
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

   
    train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=num_epochs,
        device=device
    )

    
    model.load_state_dict(torch.load("best_vit_model.pth"))
    model = model.to(device)

    
    predict_and_create_submission(
        model=model,
        test_loader=test_loader,
        device=device,
        submission_file="sample_submission.csv"
    )

if __name__ == "__main__":
    main()


Penggunaan device: cpu
Jumlah data pelatihan: 4294
Jumlah label pelatihan: 4294
Ukuran dataset train: 3435, Ukuran dataset val: 859
Jumlah data test: 2152


c:\Users\ijalg\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ijalg\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\ijalg\AppData\Local\Temp\ipykernel_21276\1894764517.py:180: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
c:\Users\ijalg\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\grad_sca

Epoch 1/5
----------


Training:   0%|          | 0/108 [00:00<?, ?it/s]C:\Users\ijalg\AppData\Local\Temp\ipykernel_21276\1894764517.py:199: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\ijalg\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
Validation:   0%|          | 0/27 [00:00<?, ?it/s]         C:\Users\ijalg\AppData\Local\Temp\ipykernel_21276\1894764517.py:223: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Train Loss: 0.1435 Val Loss: 0.0504 Val Acc: 0.9825
Model terbaik disimpan!
Epoch 2/5
----------


...

Train Loss: 0.0152 Val Loss: 0.0181 Val Acc: 0.9942
Model terbaik disimpan!
Epoch 4/5
----------


Train Loss: 0.0024 Val Loss: 0.0139 Val Acc: 0.9942
Epoch 5/5
----------


C:\Users\ijalg\AppData\Local\Temp\ipykernel_21276\1894764517.py:331: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_vit_model.pth"))


Train Loss: 0.0006 Val Loss: 0.0137 Val Acc: 0.9942
Pelatihan selesai!


File sample_submission.csv berhasil dibuat!
